# Milestone 1 lab, notebook 3 of 8: The number by hand

## Where we are

| Notebook | What you do |
|---|---|
| 1. The question and five green checks | Meet Lotus Retails and the Monday question. Run the given pipeline for one night. Read its five checks |
| 2. Airflow runs it | Let Airflow run the same night. Read the DAG: its tasks and its arrows |
| **3. The number by hand** (you are here) | Add up one store's receipts yourself. Find out why the two numbers differ |
| 4. Who is hurt | For each table: who uses it, and what happens to them when it is wrong |
| 5. The check card | Meet the page you fill in before any check. Say what one row means |
| 6. Read the table first | Count the rows, find the smallest and largest values, count the empty cells |
| 7. Six questions for any table | Sort your worries under six questions. See which ones the five checks answer |
| 8. The production version | The same logic as a tested program |

**About ten minutes.** This is the notebook the milestone is named after.

In [1]:
import csv

from lab_helpers import LANDING, money, save_result, show_file, sql

path = LANDING / "sales_lines" / "sales_lines_S03_2026-08-24.csv"
with path.open(newline="") as f:
    lines = list(csv.DictReader(f))

receipts = {line["receipt_id"] for line in lines}
print(f"Store S03, Monday 24 August: {len(lines)} lines on {len(receipts)} receipts.")

Store S03, Monday 24 August: 113 lines on 34 receipts.


## Step 1: Do not trust it. Add it up

The pipeline says store `S03` sold **$796.18** on Monday. Five checks are green.

You have the file the store sent. So work the number out yourself, with plain Python and no pipeline. It is 113 lines: small enough to believe.

First, look at one receipt.

In [2]:
first = lines[0]["receipt_id"]
print("Receipt", first)
for line in lines:
    if line["receipt_id"] == first:
        print(
            f"  {line['product_id']}  {line['quantity']:>7} x {money(int(line['unit_price_cents'])):>6}"
            f" = {money(int(line['amount_cents'])):>7}   {line['line_type']}"
        )

Receipt S03-T2-004485
  P-038    2.000 x  $3.99 =   $7.98   sale
  P-003    2.000 x  $3.49 =   $6.98   sale


## Step 2: Sales, less refunds

Net sales are the money taken for sales, less the money given back for refunds. Do it in two parts, as a person with a pile of receipts would.

First the sales.

In [3]:
sales = sum(int(line["amount_cents"]) for line in lines if line["line_type"] == "sale")
print("Money taken for sales:", money(sales))

Money taken for sales: $787.28


Then the refunds. There are only two, so print them.

In [4]:
refund_lines = [line for line in lines if line["line_type"] == "refund"]
for line in refund_lines:
    print(f"  {line['line_id']}  product {line['product_id']}  money given back: {money(abs(int(line['amount_cents'])))}")

refunds = sum(abs(int(line["amount_cents"])) for line in refund_lines)
by_hand = sales - refunds
print()
print("Money given back:    ", money(refunds))
print("Net sales, by hand:  ", money(by_hand))

  S03-0824-000003  product P-032  money given back: $0.59
  S03-0824-000008  product P-024  money given back: $9.49

Money given back:     $10.08
Net sales, by hand:   $777.20


## Step 3: Hold it against the pipeline

In [5]:
from_pipeline = int(
    sql("""
        SELECT net_sales_cents FROM output.daily_store_sales
        WHERE store_id = 'S03' AND sales_date = '2026-08-24'
    """).iloc[0, 0]
)
difference = from_pipeline - by_hand

print("The pipeline says:", money(from_pipeline))
print("By hand:          ", money(by_hand))
print("Difference:       ", money(difference))

The pipeline says: $796.18
By hand:           $777.20
Difference:        $18.98


**The two numbers differ by $18.98.** No task failed. No check failed.

One of them is wrong. Which?

## Step 4: Find the cause

A difference is a clue. $18.98 is exactly twice $9.49, and $9.49 is one of the two refunds.

Look at the two refunds again. This time print the raw columns, as the file has them.

In [6]:
for line in refund_lines:
    print(
        f"{line['line_id']}  {line['line_type']}  quantity {line['quantity']:>7}"
        f"  amount_cents {line['amount_cents']:>5}"
    )

S03-0824-000003  refund  quantity  -1.000  amount_cents   -59
S03-0824-000008  refund  quantity  -1.000  amount_cents   949


There it is.

- The first refund has `amount_cents = -59`. It is negative, so it takes money away. Right.
- The second has `amount_cents = 949`. It is a refund, its quantity is −1, and its amount is **positive**. The minus sign was lost on the way out of the store.

You read `line_type` and took both refunds away. The pipeline does not read `line_type`. This is its rule.

In [7]:
from lotus_pipeline import publish

print(publish.DAILY_STORE_SALES)


CREATE TABLE output.daily_store_sales AS
SELECT sales_date,
       store_id,
       count(DISTINCT receipt_id) AS receipts,
       count(*)                   AS lines,
       sum(amount_cents)          AS net_sales_cents
FROM clean.sales_lines
GROUP BY sales_date, store_id



`sum(amount_cents)`. The pipeline trusts the sign. So it **added** $9.49 where it should have taken $9.49 away. The error is twice the refund: $18.98.

## Step 5: A third count

Two counts disagree. A third would settle it.

Each till adds up its own day when it is closed. That total never passes through the program that writes the sales file. The store sends it in a second file, `till_totals`.

In [8]:
tills = show_file("till_totals", "till_totals_S03_2026-08-24.csv")
print("The tills' own total:", money(sum(int(cents) for cents in tills["net_total_cents"])))
tills

The tills' own total: $777.20


,business_date,store_id,till_id,receipts,gross_sales_cents,refunds_cents,net_total_cents,closed_at
0,2026-08-24,S03,T1,18,42967,1008,41959,2026-08-24 21:06:00
1,2026-08-24,S03,T2,16,35761,0,35761,2026-08-24 21:09:00


$777.20. The tills agree with you. **The pipeline is wrong.**

Nobody reads `till_totals` today. Remember this table.

## Step 6: Why did five checks pass?

Take the bad row, `S03-0824-000008`, and ask each check what it saw.

In [9]:
bad = next(line for line in refund_lines if int(line["amount_cents"]) > 0)
times = int(sql("SELECT count(*) FROM clean.sales_lines WHERE line_id = ?", (bad["line_id"],)).iloc[0, 0])

print(f"1. The table is not empty          S03 has {len(lines)} lines that day        -> pass")
print(f"2. The key is not null             line_id is {bad['line_id']}     -> pass")
print(f"3. The key is unique               it is in the table {times} time            -> pass")
print(f"4. The status is an allowed value  line_type is '{bad['line_type']}'            -> pass")
print(f"5. The total is positive           the store's day is {money(from_pipeline)}       -> pass")

1. The table is not empty          S03 has 113 lines that day        -> pass
2. The key is not null             line_id is S03-0824-000008     -> pass
3. The key is unique               it is in the table 1 time            -> pass
4. The status is an allowed value  line_type is 'refund'            -> pass
5. The total is positive           the store's day is $796.18       -> pass


Every check is right about what it looked at. None of them asks what a refund **is**.

"A refund gives money back" is a rule of the business. The five checks test only the shape of the data: is the key there, is it unique, is the status on the list.

## Step 7: What it costs, and its name

The bonus is 1% of net sales.

In [10]:
from lotus_pipeline.report import bonus_cents

paid, right = bonus_cents(from_pipeline), bonus_cents(by_hand)
print(f"Bonus from the pipeline's number: {money(paid)}")
print(f"Bonus from the right number:      {money(right)}")
print(f"Overpaid for this one day:        {money(paid - right)}")

Bonus from the pipeline's number: $7.96
Bonus from the right number:      $7.77
Overpaid for this one day:        $0.19


Nineteen cents. Tonight the error is small.

But look at what let it through. The checks cannot see a refund of **any** size, in any store, on any night. And the head of retail operations signs the number because the checks are green.

This kind of failure has a name: a **silent failure**. The data is wrong and there is no error. Every task succeeds, and a wrong number reaches the reader.

Its opposite is a loud failure: a task stops with an error, and everybody knows. A loud failure is the better one to have.

Save what you found. A test will hold the production code against it.

In [11]:
save_result(
    "answer_key_by_hand",
    {
        "store_id": "S03",
        "day": "2026-08-24",
        "by_hand_cents": by_hand,
        "pipeline_cents": from_pipeline,
        "difference_cents": difference,
        "bad_line_id": bad["line_id"],
    },
)

Saved notebooks/results/answer_key_by_hand.json


## What you now know

- Store `S03` sold $777.20 on Monday. The pipeline says $796.18.
- The cause is one row: a refund of $9.49 stored as a positive amount.
- All five checks passed, and each was right about what it looked at.
- A silent failure is wrong data with no error.
- **You found it by luck.** You happened to add up `S03`. Nobody added up the other five stores.

**Say it in an interview:** "I added up one store's receipts by hand and got $777.20. The pipeline said $796.18 with five green checks. A refund was stored as a positive amount, so it was added and not taken away. That is a silent failure: no task failed, and a wrong number reached the person who signs the bonus."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "One bad row, one wrong number". Silent and loud failures are in [Schema Evolution & Data Contracts](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/schema-evolution-and-data-contracts), section "What breaks in the job".

Next: [notebook 4](04_who_is_hurt.ipynb). Before you write a check, ask who it is for.